# Connectivity Hyperalignment (CHA)

Connectivity Hyperalignment (CHA) estimates hyperalignment transformations from functional-connectivity profiles. The workflow starts from prepared response data, computes functional connectivity (FC) matrices for each subject, constructs a shared connectivity space, estimates T matrices from that space, and applies those matrices to response data or other feature matrices in the same spatial space.

The examples on this page use response arrays saved as `.npy` files with shape `time points x vertices/voxels` in the HA work directory. FC outputs are saved under `<seed_step>/connectivity/<seed_structure>` within each subject folder. Each FC matrix has shape `n_target x n_seed`; each column stores one seed's connectivity profile across target nodes.

## Script-Based Workflow

The script workflow contains five stages. The first stage prepares three kinds of searchlights for CHA: alignment searchlights, seed definitions, and target definitions. The second stage builds FC files through `ha.fc_build`. The last three stages reuse the RHA hyperalignment functions with `modality="connectivity"` for template and transform estimation.

```{important}
In a real workflow, the Template, T Matrix, and Alignment steps should <span style="color: red; font-weight: bold;">NOT</span> all be estimated from exactly the same data. The script examples below estimate common spaces and T matrices from `run="01"`, then apply the transformations to `run="02"`.
```

**Required script inputs**

| Item | Meaning |
|---|---|
| `work_dir` | Root HA work directory. It contains subject folders, `logs`, `masks`, FC outputs, common-space outputs, transform outputs, and aligned outputs. |
| `sub_list` | Subject folder names used for FC construction, common-space construction, T matrix calculation, and alignment. |

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path

import nibabel as nib
import numpy as np
import neuroboros as nb

from fmriha import gensls, ha
from fmriha.gensls.dsample import downsample_nifti_volume

In [ ]:
work_dir = Path("/path/to/ha_workdir")
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

### (1) Searchlight Preparation

CHA uses searchlights in three roles:

- Alignment searchlights are used by `ha.cspace_sls` and `ha.xform_sls` in the {ref}`RHA common-space construction <rha-common-space-construction>` and {ref}`T matrix calculation <rha-t-matrix-calculation>` stages. They define local neighborhoods over the seed vertex/voxel axis and match the RHA searchlights.

- Seed definitions are passed to `ha.fc_build(seed_sls=...)`. They usually contain one vertex/voxel per seed, generated from the searchlight center. In each searchlight list, the first element is the center point.

- Target definitions are passed to `ha.fc_build(target_sls=...)`. They are usually coarser searchlights or downsampled neighborhoods used to build connectivity profiles.

Shared searchlight functions are documented in the {ref}`RHA searchlight preparation section <rha-searchlight-preparation>`. CHA changes how the returned searchlights are assigned to seed and target roles.

**Cortical**

For cortical CHA, the alignment searchlights use the same `nb.sls` call described in {ref}`RHA searchlight preparation <rha-searchlight-preparation>`, with `mask=True` and `return_dists=True`. CHA then reuses those returned neighborhoods in two additional ways: seed definitions are built with `[np.array([idx[0]]) for idx in sls_surf[lr]]`, where `idx[0]` selects the center point of each searchlight, and connectivity targets are generated with a lower-resolution `center_space` and `return_dists=False`.



In [ ]:
surface_sls_lr = {
    lr: nb.sls(
        lr,
        radius=20,
        space="onavg-ico32",
        mask=True,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf = {lr: surface_sls_lr[lr][0] for lr in surface_sls_lr}
dists_surf = {lr: surface_sls_lr[lr][1] for lr in surface_sls_lr}

sls_surf_seed = {
    lr: [np.array([idx[0]]) for idx in sls_surf[lr]]
    for lr in "lr"
}

sls_surf_target = {
    lr: nb.sls(
        lr,
        radius=13,
        space="onavg-ico32",
        center_space="onavg-ico8",
        mask=True,
        return_dists=False,
    )
    for lr in "lr"
}


**Subcortical**

For CIFTI subcortical CHA, dense masks are created with the same `gensls.downsample_cifti_subcortical` settings described in {ref}`RHA searchlight preparation <rha-searchlight-preparation>`: `N=None` and `sls_type="seed"`. Dense volume searchlights are generated with the same `gensls.generate_searchlight_vol` call as RHA, and the center point of each dense searchlight is then used as a single-voxel seed. In each searchlight list, this center point is stored as the first element. CHA adds target neighborhoods by calling `gensls.downsample_cifti_subcortical` with `N=1994` and `sls_type="not_seed"` to create target centers, followed by `gensls.generate_searchlight_vol` with dense masks, those center masks, and `threshold=0` for subcortical targets.



In [ ]:
reference_cifti = "/path/to/reference_run01.dtseries.nii"

# Alignment searchlights: local neighborhoods used to build CHA common spaces and T matrices.
mask_dense_subcortical = {
    lr: gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=None,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

sls_info_sub_align = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_subcortical[lr],
        mask_center=None,
        radius=4,
        threshold=0.2,
        njobs=10,
        verbose=5,
    )
    for lr in ["l", "r", "brain_stem"]
}

sls_sub_align = {
    f"subcortical-{lr}": sls_info_sub_align[lr]["sls_idx"]
    for lr in ["l", "r", "brain_stem"]
}

dists_sub_align = {
    f"subcortical-{lr}": sls_info_sub_align[lr]["dists"]
    for lr in ["l", "r", "brain_stem"]
}

# Seed definitions: one seed per alignment searchlight center for FC construction.
sls_sub_seed = {
    f"subcortical-{lr}": np.array(
        [sl[0] for sl in sls_sub_align[f"subcortical-{lr}"]],
        dtype=np.int32,
    )
    for lr in ["l", "r", "brain_stem"]
}

# Target searchlights: downsampled target neighborhoods used as FC profile nodes.
mask_dense_sub_target = {
    f"subcortical-{lr}": gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=1994,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

mask_center_sub_target = {
    f"subcortical-{lr}": gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=1994,
        mask3d_out=None,
        sls_type="not_seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

sls_info_sub_target = {
    f"subcortical-{lr}": gensls.generate_searchlight_vol(
        mask_dense=mask_dense_sub_target[f"subcortical-{lr}"],
        mask_center=mask_center_sub_target[f"subcortical-{lr}"],
        radius=4,
        threshold=0,
        njobs=10,
        verbose=1,
    )
    for lr in ["l", "r", "brain_stem"]
}

sls_sub_target = {
    f"subcortical-{lr}": sls_info_sub_target[f"subcortical-{lr}"]["sls_idx"]
    for lr in ["l", "r", "brain_stem"]
}


**Volume ROI**

For a NIfTI ROI, the dense ROI mask is used for alignment searchlights and seed extraction. Target centers are generated by `downsample_nifti_volume`, then passed to `gensls.generate_searchlight_vol`; see {ref}`RHA searchlight preparation <rha-searchlight-preparation>`.

Parameters of `downsample_nifti_volume`:

| Parameter | Default | Meaning |
|---|---|---|
| `mask_data` | Required | Boolean or binary 3D ROI mask. Non-zero voxels are candidate target centers. |
| `N` | `None` | Number of target centers to keep. With `None`, the function keeps `ceil(0.062 * n_mask_voxels)`. Values larger than the mask size are clipped to the mask size. |
| `voxel_sizes` | `None` | Optional voxel spacing used when sampling centers. With `None`, voxel-index coordinates are used. |
| `bins_per_axis` | `16` | Number of bins along each axis for near-uniform spatial sampling. |
| `alpha_count` | `0.005` | Weight controlling how strongly bin-count uniformity affects the base selection. |
| `modulus` | `8` | Grid-residue modulus used during candidate selection. |
| `beam_width` | `12` | Number of residue candidates retained during search. |
| `max_offsets` | `64` | Maximum number of residue offsets tested. |
| `rng_seed` | `616` | Random seed used by farthest-point sampling when tie-breaking or growing the selection. |



In [ ]:
roi_name = "mPFC"
nifti_mask_path = "/path/to/mPFC_mask.nii.gz"

# Alignment searchlights: local ROI neighborhoods used to build CHA common spaces and T matrices.
mask_dense_roi = nib.load(nifti_mask_path).get_fdata().astype(bool)

sls_info_roi_align = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=None,
    radius=4,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi_align = {f"volume-{roi_name}": sls_info_roi_align["sls_idx"]}
dists_roi_align = {f"volume-{roi_name}": sls_info_roi_align["dists"]}

# Seed definitions: one seed per alignment searchlight center for FC construction.
sls_roi_seed = {
    f"volume-{roi_name}": np.array(
        [sl[0] for sl in sls_info_roi_align["sls_idx"]],
        dtype=np.int32,
    )
}

# Target searchlights: downsampled ROI target neighborhoods used as FC profile nodes.
mask_center_roi_target = downsample_nifti_volume(mask_dense_roi)

sls_info_roi_target = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=mask_center_roi_target,
    radius=4,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi_target = {f"volume-{roi_name}": sls_info_roi_target["sls_idx"]}


(functional-connectivity-calculation)=

### (2) Functional Connectivity Calculation

`ha.fc_build` converts prepared response data into CHA connectivity profiles. For each subject, it loads one seed response file, converts seed searchlights into seed-node time series, loads one target response file per target structure, converts target searchlights into target-node time series, concatenates target nodes across structures, and saves a target-by-seed FC matrix.

Parameters of `ha.fc_build`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folders included in FC construction. |
| `seed_step` | Required | Step folder containing seed response files, usually `resample` or `align`. |
| `seed_modality` | Required | Modality folder containing seed data, usually `response`. |
| `seed_structure` | Required | One seed structure folder. The function accepts a string or a one-element list, such as `hemi-L` or `volume-mPFC`. |
| `seed_file_filter` | Required | BIDS-style filters selecting exactly one seed `.npy` file per subject. |
| `seed_sls` | Required | Seed searchlight definitions. In CHA these are usually single-vertex/voxel seeds taken from searchlight centers. |
| `target_step` | Required | Step folder containing target response files, usually `resample` or `align`. |
| `target_modality` | Required | Modality folder containing target data, usually `response`. |
| `target_structure` | Required | List of target structure folders. Target nodes are concatenated in this order. |
| `target_file_filter` | Required | BIDS-style filters selecting exactly one target `.npy` file per subject and target structure. |
| `target_sls` | Required | Target searchlight definitions matching `target_structure`. |
| `zscore` | `True` | If `True`, z-scores each seed connectivity profile across targets after correlation calculation. |
| `njobs` | `5` | Number of joblib workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Numeric dtype used for node time series and saved FC arrays. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `00001` | Identifier appended to the progress log filename. |
| `suffix` | `None` | Optional label appended to FC output filenames as `suffix-<suffix>`. |
| `get_name` | `False` | If `True`, returns the saved FC filenames. |
| `dask` | `False` | If `True`, uses the current Dask client for FC construction. |
| `batch_size_dask` | `30` | Number of subjects submitted in each Dask batch. |

FC output filenames include `desc-fc-zscore` when `zscore=True` and `desc-fc-raw` when `zscore=False`. Common-space and transformation stages should select these files with filters such as `{"desc": "fc-zscore", "suffix": "lowall"}`.

**Cortical**

This example builds cortical seed FC files for `hemi-L` and `hemi-R`, using downsampled cortical target searchlights from both hemispheres.



In [ ]:
response_filter = {"run": "01"}

for seed_struct in ["hemi-L", "hemi-R"]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=response_filter,
        seed_sls=sls_surf_seed,
        target_step="resample",
        target_modality="response",
        target_structure=["hemi-L", "hemi-R"],
        target_file_filter=response_filter,
        target_sls=sls_surf_target,
        njobs=32,
    )

**Dask version**

To run CHA FC construction with Dask, first create a Dask cluster and connect a client. The FC inputs remain the same as in the single-machine version; the only changes to `ha.fc_build` are setting `dask=True` and, if needed, `batch_size_dask`. The example below uses a SLURM cluster; for detailed Dask jobqueue cluster parameters, see [https://jobqueue.dask.org/en/latest/clusters-api.html](https://jobqueue.dask.org/en/latest/clusters-api.html). Replace the example SLURM parameters below with values appropriate for your computing environment.

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

dask_params = {
    "queue": "your_queue",
    "project": "CHA_DASK_DEMO",
    "account": "your_account",
    "cores": 20,
    "memory": "140GB",
    "jobs_num": 4,
    "processes": 1,
    "walltime": "2-00:00:00",
    "scheduler_options": {"dashboard_address": ":1234"},
}

slurm_log_path = Path(work_dir) / "logs" / "slurm_out"
os.makedirs(slurm_log_path, exist_ok=True)

cluster = SLURMCluster(
    queue=dask_params["queue"],
    cores=dask_params["cores"],
    memory=dask_params["memory"],
    processes=dask_params["processes"],
    account=dask_params["account"],
    walltime=dask_params["walltime"],
    scheduler_options=dask_params["scheduler_options"],
    job_name=dask_params["project"],
    log_directory=str(slurm_log_path),
    worker_extra_args=["--death-timeout", "6000"],
)
cluster.scale(jobs=dask_params["jobs_num"])
client = Client(cluster)  # connect client

for seed_struct in ["hemi-L", "hemi-R"]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=response_filter,
        seed_sls=sls_surf_seed,
        target_step="resample",
        target_modality="response",
        target_structure=["hemi-L", "hemi-R"],
        target_file_filter=response_filter,
        target_sls=sls_surf_target,
        dask=True,  # bool
        batch_size_dask=30,
    )

**Subcortical**

This example uses subcortical structures as seeds and uses the whole brain, including both cortical and subcortical neighborhoods, as the target set.

In [ ]:
target_sls_whole = {
    "hemi-L": sls_surf_target["l"],
    "hemi-R": sls_surf_target["r"],
    "subcortical-l": sls_sub_target["subcortical-l"],
    "subcortical-r": sls_sub_target["subcortical-r"],
    "subcortical-brain_stem": sls_sub_target["subcortical-brain_stem"],
}

target_structures_whole = [
    "hemi-L",
    "hemi-R",
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]

for seed_struct in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=response_filter,
        seed_sls=sls_sub_seed,
        target_step="resample",
        target_modality="response",
        target_structure=target_structures_whole,
        target_file_filter=response_filter,
        target_sls=target_sls_whole,
        njobs=32,
    )


If the analysis focuses on cortical seeds but the target profiles should include the whole brain, use `hemi-L` and `hemi-R` as seeds while keeping the same cortical plus subcortical target set.



In [ ]:
for seed_struct in ["hemi-L", "hemi-R"]:
    ha.fc_build(
        work_dir,
        sub_list,
        seed_step="resample",
        seed_modality="response",
        seed_structure=seed_struct,
        seed_file_filter=response_filter,
        seed_sls=sls_surf_seed,  # Surface sls
        target_step="resample",
        target_modality="response",
        target_structure=target_structures_whole,
        target_file_filter=response_filter,
        target_sls=target_sls_whole,    # Whole sls
        njobs=32,
    )

For the Dask version, use the same running Dask client and set `dask=True` in each `ha.fc_build` call. Adjust `batch_size_dask` as needed for the number of subjects and worker memory.


**Volume ROI**

This example builds ROI seed FC files with ROI target neighborhoods. The target list can be expanded to include cortical or subcortical targets if the study design requires cross-structure connectivity profiles.



In [ ]:
ha.fc_build(
    work_dir,
    sub_list,
    seed_step="resample",
    seed_modality="response",
    seed_structure=f"volume-{roi_name}",
    seed_file_filter=response_filter,
    seed_sls=sls_roi_seed,
    target_step="resample",
    target_modality="response",
    target_structure=[f"volume-{roi_name}"],
    target_file_filter=response_filter,
    target_sls=sls_roi_target,
    njobs=32,
)

For the Dask version, use the same running Dask client and set `dask=True` in the `ha.fc_build` call. Adjust `batch_size_dask` as needed for the number of subjects and worker memory.

### (3) Common Space Construction

`ha.cspace_sls` is shared with RHA; see {ref}`RHA common-space construction <rha-common-space-construction>`. Its algorithmic arguments have the same meaning as in RHA, including `cspace_kind`, `common_topography`, `weight`, `demean`, `start_sub`, `chunk_size`, `dtype`, and `scope`.

The CHA call differs in the data it reads. `sls`, `dists`, and `radius` describe alignment searchlights over the seed axis of FC matrices. `modality` is set to `connectivity`, `step` points to the step where `ha.fc_build` saved FC files, `structure_name` names the seed structures whose FC files were built, and `**file_filters` select FC outputs, commonly `desc="fc-zscore"` plus a `suffix` filter when a suffix was saved. `task_name` should use a CHA label such as `chaPRcortical`, `chaPRsubcortical`, or `chaPRroi`.

**Cortical**



In [ ]:
fc_file_filter = {"desc": "fc-zscore"}

ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    task_name="chaPRcortical",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="connectivity",  # FC
    structure_name=["hemi-L", "hemi-R"],
    **fc_file_filter,
)

**Dask version**

Reuse the cluster/client created in the FC step. Compared with the single-machine version, the common-space call sets `dask=True` and passes `cluster=cluster`. If you have not previously run the FC Dask workflow above, create and connect to a Dask cluster first.

In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    task_name="chaPRcortical",
    cspace_kind="procrustes",
    step="resample",
    modality="connectivity",  # FC
    structure_name=["hemi-L", "hemi-R"],
    dask=True,  # bool
    cluster=cluster,  # cluster configuration
    **fc_file_filter,
)


**Subcortical**



In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=4,
    sub_list=sub_list,
    task_name="chaPRsubcortical",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="connectivity",  # FC
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    **fc_file_filter,
)

For the Dask version, the only difference is setting `dask=True` and passing `cluster=cluster` in the `ha.cspace_sls` call.


**Volume ROI**



In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=4,
    sub_list=sub_list,
    task_name="chaPRroi",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="connectivity",  # FC
    structure_name=[f"volume-{roi_name}"],
    **fc_file_filter,
)

For the Dask version, the only difference is setting `dask=True` and passing `cluster=cluster` in the `ha.cspace_sls` call.


### (4) T matrix calculation

This stage uses `ha.xform_sls`; see {ref}`RHA T matrix calculation <rha-t-matrix-calculation>`. Its Procrustes and aggregation arguments have the same meaning as in RHA, including `reflection`, `scaling`, `xform_weight`, `concatenated`, `chunk_size`, `dtype`, `njobs`, and `scope`.

The CHA call estimates sparse T matrices by aligning each subject's connectivity files to the CHA common space. Use the same alignment `sls`, `dists`, and `radius` used for CHA common-space construction. Set `source_step` to the step where FC files were saved, set `modality="connectivity"`, select the seed structures in `structure_name`, match `task_name` to the CHA common-space task, and use `cspace_desc` such as `{"alg": "procrustes"}` when needed to identify one common-space file. Source `**file_filters` should select the FC files, commonly `desc="fc-zscore"` plus the FC suffix.

**Cortical**



In [ ]:
common_space_filter = {"alg": "procrustes"}

ha.xform_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",  # FC
    structure_name=["hemi-L", "hemi-R"],
    task_name="chaPRcortical",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

**Dask version**

Reuse the cluster/client created in the FC step. Compared with the single-machine version, the T matrix call only sets `dask=True`. If you have not previously run a Dask workflow above, create a Dask cluster and connect with `Client(cluster)` first.

In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",  # FC
    structure_name=["hemi-L", "hemi-R"],
    task_name="chaPRcortical",
    cspace_desc=common_space_filter,
    dask=True,  # bool
    **fc_file_filter,
)


**Subcortical**



In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",  # FC
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="chaPRsubcortical",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

For the Dask version, the only difference is to set `dask=True` in the `ha.xform_sls` call.


**Volume ROI**



In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",  # FC
    structure_name=[f"volume-{roi_name}"],
    task_name="chaPRroi",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

For the Dask version, the only difference is to set `dask=True` in the `ha.xform_sls` call.

**Additional Transformation Functions**

`ha.xform_sls_con_mega` and `ha.xform_sls_ucon_mega` can also be used in CHA when chunk-wise sparse accumulation is preferred. Their parameter meanings follow the {ref}`RHA transformation section <rha-t-matrix-calculation>`. In CHA, set `modality="connectivity"`, select the FC source files with `**file_filters`, and replace `sls`, `dists`, `radius`, `structure_name`, and `task_name` for cortical, subcortical, or ROI runs.

For most CHA analyses, `ha.xform_sls` remains the recommended entry point; see {ref}`RHA T matrix calculation <rha-t-matrix-calculation>`.

**Cortical**


In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=["hemi-L", "hemi-R"],
    task_name="chaPRcortical",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=20,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=["hemi-L", "hemi-R"],
    task_name="chaPRcortical",
    cspace_kind="procrustes",
    njobs=32,
    **fc_file_filter,
)


**Subcortical**



In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="chaPRsubcortical",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_sub_align,
    dists=dists_sub_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="chaPRsubcortical",
    cspace_kind="procrustes",
    njobs=32,
    **fc_file_filter,
)


**Volume ROI**



In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[f"volume-{roi_name}"],
    task_name="chaPRroi",
    cspace_desc=common_space_filter,
    njobs=32,
    **fc_file_filter,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_roi_align,
    dists=dists_roi_align,
    radius=4,
    sub_list=sub_list,
    source_step="resample",
    modality="connectivity",
    structure_name=[f"volume-{roi_name}"],
    task_name="chaPRroi",
    cspace_kind="procrustes",
    njobs=32,
    **fc_file_filter,
)


### (5) Alignment

`ha.align_pipe` is shared with RHA; see {ref}`RHA Alignment <rha-alignment>`. Its source-file selection and matrix-application behavior stay the same. In a typical CHA workflow, the T matrices are estimated from connectivity files, then applied to response data from the same feature space.

The CHA call usually keeps `source_modality="response"` and uses `source_name_filter` to select the response data to align, often a held-out run such as `run="02"`. Set `xform_modality="connectivity"`, because CHA T matrices are saved under `xforms/connectivity`; set `xform_structure_name=None` so transform lookup follows the selected source structure; and set `xform_name_filter` to the CHA transform task, such as `{"task": "chaPRcortical"}`.

**Cortical**



In [ ]:
align_source_filter = {"run": "02"}

for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="connectivity",  # T matrix from connectivity
        xform_structure_name=None,
        xform_name_filter={"task": "chaPRcortical"},
        njobs=8,
    )

**Dask version**

Reuse the same running cluster/client for alignment. Set `dask=True`; `batch_size_dask` controls how many subjects are submitted per Dask batch. Close the client and cluster after the Dask workflow finishes.

In [ ]:
for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="connectivity",  # T matrix from connectivity
        xform_structure_name=None,
        xform_name_filter={"task": "chaPRcortical"},
        dask=True,  # bool
        batch_size_dask=30,
    )

client.close()
cluster.close()


**Subcortical**



In [ ]:
for structure in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter=align_source_filter,
        xform_modality="connectivity",  # T matrix from connectivity
        xform_structure_name=None,
        xform_name_filter={"task": "chaPRsubcortical"},
        njobs=8,
    )

For the Dask version, the only difference is setting the `dask` parameter to `True` and setting `batch_size_dask` as needed in the `ha.align_pipe` call.


**Volume ROI**



In [ ]:
ha.align_pipe(
    work_dir,
    sub_list,
    source_step="resample",
    source_modality="response",
    source_structure_name=f"volume-{roi_name}",
    source_name_filter=align_source_filter,
    xform_modality="connectivity",  # T matrix from connectivity
    xform_structure_name=None,
    xform_name_filter={"task": "chaPRroi"},
    njobs=8,
)

For the Dask version, the only difference is setting the `dask` parameter to `True` and setting `batch_size_dask` as needed in the `ha.align_pipe` call.

## GUI Workflow

Launch the graphical interface with `gui()`. Before configuring CHA stages, fill **Work Dir** at the top of the GUI. 

In [ ]:
from fmriha import gui
gui()

```{image} pic/data_prep/gui.png
:width: 800px
```

First, click the **n_jobs Parameters** button and specify the number of parallel CPU cores for the **Connectivity Calculation**, **Template Calculation**, **T matrix calculation**, and **Alignment** steps. These values are passed to the `njobs` arguments of `ha.fc_build`, `ha.cspace_sls`, `ha.xform_sls`, and `ha.align_pipe`; see [Functional Connectivity Calculation](#functional-connectivity-calculation), {ref}`RHA common-space construction <rha-common-space-construction>`, {ref}`RHA T matrix calculation <rha-t-matrix-calculation>`, and {ref}`RHA Alignment <rha-alignment>`. They have defaults, so editing them is optional unless you want to change parallelism.

```{image} pic/cha/gui_njobs.png
:width: 800px
```

First, click the **Space and Searchlight Configuration** button. The alignment searchlight settings have the same meanings as on the [Response Hyperalignment page](rha.ipynb): surface **Space** and **Surface Searchlight Radius** are required for surface seed/alignment structures; **Volume Searchlight Radius** and **Threshold** are required for volume structures; **CIFTI Reference** is required for CIFTI subcortical structures; **NIFTI Mask** and **NIFTI ROI Name** are required for NIFTI ROI structures; **Custom Mask** is optional. Only `.pkl` Surface mask files are supported; retained surface positions must be marked as `True`, and positions to mask out must be marked as `False`. Detailed instructions can be found in the [RHA GUI searchlight section](rha.ipynb#gui-workflow).

After configuring the searchlights, return to the main page and enable the **Connectivity Calculation** section. This panel maps to `ha.fc_build`.

Required fields in this panel are **Seed Structure**, **Target Structure**, and at least one subject source in **Data List**. You must check at least one of **Template Data**, **Transformation Data**, **Alignment Data**, or **Other Subjects**; if **Other Subjects** is checked, the subject list next to it is required. **Seed File Filter** and **Target File Filter** are required unless the selected seed and target folders each contain exactly one matching `.npy` file per subject. **Seed Step**, **Seed Modality**, **Target Step**, and **Target Modality** have defaults but should match where the source response files are stored. **Target Searchlight Space** and the target searchlight radii have defaults; the surface radius is used for surface targets and the volume radius is used for volume targets. **Zscore** and **Float Type** have defaults, and **Suffix** is optional. Parameter meanings for `ha.fc_build` are described in the [Functional Connectivity Calculation section](#functional-connectivity-calculation).

Here, we still use `run-01` to construct the Common Template and T matrix, and then apply the T matrix to `run-02`. Therefore, in this section, the File Filter used corresponds to each subject's `run-01`, and the specific Data List consists of the corresponding `Template Data` and `Transformation Data`.

```{image} pic/cha/gui_fc.png
:width: 800px
```

The subsequent procedures for **Template Calculation**, **T matrix calculation**, and **Alignment** use the same GUI panels described in RHA, so their parameter meanings for `ha.cspace_sls`, `ha.xform_sls`, and `ha.align_pipe` follow the {ref}`RHA common-space construction <rha-common-space-construction>`, {ref}`T matrix calculation <rha-t-matrix-calculation>`, and {ref}`Alignment <rha-alignment>` sections. For CHA Template and T matrix calculation, required fields are **Structure**, **Task Name**, **Step**, **Modality**, and **Sub List** when Data Preparation is not being run in the same GUI workflow. Set **Modality** to `connectivity`. **File Filter** is required when several FC outputs could match the same structure; in the screenshot this is `space=surf`, matching the FC files built from `space-surf`. Add more filters, such as `desc` or `suffix`, when needed.

For CHA T matrix calculation, **Common Space** is also required, typically `alg=procrustes`, so the GUI can select the common-space file. For Alignment, required fields are **Source Step**, **Source Modality**, **Source Structure**, **Source Name Filter**, **Transform Modality**, **Transform Name Filter**, and **Sub List** when Data Preparation is not being run in the same GUI workflow. The source data are usually response data from the held-out run, while **Transform Modality** should be `connectivity` so the GUI selects CHA T matrices. Fields with visible defaults such as **Chunk Size**, **Float Type**, **Scope**, and the transform option checkboxes can be left unchanged unless your workflow requires different settings; **Suffix** is optional.

```{image} pic/cha/gui_chaTmpl.png
:width: 800px
```

```{image} pic/cha/gui_chaT.png
:width: 800px
```

```{image} pic/cha/gui_chaApply.png
:width: 800px
```

Finally, just like in RHA, click the **RUN!** button to start the process, and you can click the **Monitor** button to track the execution progress.

In the next chapter, we will introduce several statistical analysis methods, such as ISC (Inter-subject Correlation) and other metrics.